# Bronze Batch Finalizer

Marks batch ingestion completion and updates watermarks.

**Purpose**: Track batch completion and data freshness  
**Pattern**: Immutable batch state records  
**Watermarks**: Enable incremental processing downstream

## ✓ Batch ID Consistency Flow

**Single Source of Truth: Ingestion generates batch_id once**

```
Ingestion (ingest_common_helpers)
    ↓ generates UUID batch_id
    ├─→ bronze_job_snapshot (batch_id)
    ├─→ bronze_api_response_log (batch_id)
    ↓
Deduplication (bronze_dedupe_raw_payload)
    ↓ uses ingestion batch_id (auto-detected or passed)
    ├─→ dedupe_tracking (first_seen_batch_id = ingestion batch_id)
    ↓
Finalization (bronze_finalize_batch)
    ↓ uses ingestion batch_id (auto-detected or passed)
    └─→ batch_metadata (batch_id = ingestion batch_id)
```

**Key Changes Made:**
1. **bronze_dedupe_raw_payload**: Now pulls batch_id from bronze_job_snapshot instead of generating new UUID
2. **bronze_finalize_batch**: Now pulls batch_id from bronze_job_snapshot instead of dedupe_tracking
3. **Result**: batch_id flows consistently from ingestion → deduplication → finalization

**Verification**: All Bronze tables now reference the same ingestion batch_id for complete lineage traceability

In [0]:
# Batch finalization parameters
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("source_name", "", "Source Name")
dbutils.widgets.text("records_ingested", "0", "Records Ingested")
dbutils.widgets.text("records_failed", "0", "Records Failed")
dbutils.widgets.text("watermark_value", "", "Watermark Value")
dbutils.widgets.dropdown("status", "success", ["success", "partial", "failed"], "Status")
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

# Get values
batch_id = dbutils.widgets.get("batch_id")
source_name = dbutils.widgets.get("source_name")
records_ingested = int(dbutils.widgets.get("records_ingested"))
records_failed = int(dbutils.widgets.get("records_failed"))
watermark_value = dbutils.widgets.get("watermark_value")
status = dbutils.widgets.get("status")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Auto-detect batch_id and source_name from latest ingestion if not provided
if not batch_id or not source_name:
    try:
        latest_batch = spark.sql("""
            SELECT batch_id, source_name
            FROM workspace.bronze.bronze_job_snapshot 
            WHERE batch_id IS NOT NULL AND source_name IS NOT NULL
            ORDER BY ingestion_timestamp DESC 
            LIMIT 1
        """).collect()
        
        if latest_batch:
            if not batch_id:
                batch_id = latest_batch[0]["batch_id"]
                print(f"📌 Auto-detected batch_id from bronze_job_snapshot: {batch_id}")
            if not source_name:
                source_name = latest_batch[0]["source_name"]
                print(f"📌 Auto-detected source_name from bronze_job_snapshot: {source_name}")
        else:
            raise ValueError("Could not find ingestion batches in bronze_job_snapshot")
    except Exception as e:
        raise ValueError(f"Could not auto-detect batch parameters: {str(e)}")

if not batch_id:
    raise ValueError("batch_id is required")
if not source_name:
    raise ValueError("source_name is required")

print(f"Finalizing batch: {batch_id}")
print(f"Source: {source_name}")
print(f"Status: {status}")

In [0]:
%sql
-- Create batch metadata table
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.batch_metadata (
  batch_id STRING,
  source_name STRING,
  records_ingested INT,
  records_failed INT,
  watermark_value STRING,
  status STRING,
  batch_start_timestamp TIMESTAMP,
  batch_end_timestamp TIMESTAMP,
  finalization_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Immutable batch execution records and watermarks'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Create batch metadata record
batch_df = spark.createDataFrame([(
    batch_id,
    source_name,
    records_ingested,
    records_failed,
    watermark_value if watermark_value else None,
    status,
    datetime.now(),
    datetime.now(),
    datetime.now()
)], schema="""
    batch_id STRING,
    source_name STRING,
    records_ingested INT,
    records_failed INT,
    watermark_value STRING,
    status STRING,
    batch_start_timestamp TIMESTAMP,
    batch_end_timestamp TIMESTAMP,
    finalization_timestamp TIMESTAMP
""")

# Append to Bronze table
target_table = f"{catalog}.{schema}.batch_metadata"
batch_df.write.mode("append").saveAsTable(target_table)

print(f"✓ Batch finalized in {target_table}")
print(f"  Batch ID: {batch_id}")
print(f"  Source: {source_name}")
print(f"  Status: {status}")
print(f"  Records ingested: {records_ingested}")
print(f"  Records failed: {records_failed}")
if watermark_value:
    print(f"  Watermark: {watermark_value}")

In [0]:
%sql
-- Create watermark tracking table
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.watermarks (
  source_name STRING,
  watermark_value STRING,
  last_batch_id STRING,
  update_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Current watermark state per source'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

In [0]:
# Update watermark if provided
if watermark_value and status == "success":
    from delta.tables import DeltaTable
    from datetime import datetime
    
    watermark_table = f"{catalog}.{schema}.watermarks"
    
    # Create watermark update record
    watermark_df = spark.createDataFrame([(
        source_name,
        watermark_value,
        batch_id,
        datetime.now()
    )], schema="source_name STRING, watermark_value STRING, last_batch_id STRING, update_timestamp TIMESTAMP")
    
    # Check if table exists
    if spark.catalog.tableExists(watermark_table):
        # Merge watermark
        delta_table = DeltaTable.forName(spark, watermark_table)
        delta_table.alias("target").merge(
            watermark_df.alias("source"),
            "target.source_name = source.source_name"
        ).whenMatchedUpdate(
            set={
                "watermark_value": "source.watermark_value",
                "last_batch_id": "source.last_batch_id",
                "update_timestamp": "source.update_timestamp"
            }
        ).whenNotMatchedInsertAll().execute()
    else:
        # First time - insert
        watermark_df.write.mode("append").saveAsTable(watermark_table)
    
    print(f"✓ Watermark updated in {watermark_table}")
    print(f"  Source: {source_name}")
    print(f"  Watermark: {watermark_value}")